In [0]:
spark.table("workspace.logistics_project.delivery_events").printSchema()

root
 |-- event_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- scheduled_datetime: timestamp (nullable = true)
 |-- actual_datetime: timestamp (nullable = true)
 |-- detention_minutes: long (nullable = true)
 |-- on_time_flag: boolean (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)



In [0]:
#Read Tables
import pyspark.sql.functions as F

delivery = spark.table("workspace.logistics_project.delivery_events")
loads = spark.table("workspace.logistics_project.loads")

In [0]:
#Join Tables
delivery_analysis = (
    delivery
    .join(loads, "load_id", "left")
)

In [0]:
#Calculate Delivery KPIs
delivery_metrics = (
    delivery_analysis
    .groupBy(
        "facility_id",
        "location_city",
        "location_state",
        "event_type"
    )
    .agg(
        F.count("event_id").alias("total_events"),

        F.sum(
            F.when(F.col("on_time_flag") == True, 1)
             .otherwise(0)
        ).alias("on_time_events"),

        F.round(
            F.avg("detention_minutes"),2
        ).alias("avg_detention_minutes"),

        F.round(
            F.sum("detention_minutes"),2
        ).alias("total_detention_minutes"),

        F.round(
            F.sum("revenue"),2
        ).alias("total_revenue")
    )
)

In [0]:
#Calculate On-Time Delivery %
delivery_metrics = delivery_metrics.withColumn(
    "on_time_rate_pct",
    F.round(
        (F.col("on_time_events") /
         F.col("total_events")) * 100,
        2
    )
)

In [0]:
#Calculate Delay Events
delivery_metrics = delivery_metrics.withColumn(
    "delayed_events",
    F.col("total_events") - F.col("on_time_events")
)

In [0]:
#Display Results
display(
    delivery_metrics.orderBy(
        F.col("on_time_rate_pct").desc()
    )
)

facility_id,location_city,location_state,event_type,total_events,on_time_events,avg_detention_minutes,total_detention_minutes,total_revenue,on_time_rate_pct,delayed_events
FAC00040,Denver,CO,Pickup,21,19,71.0,1491,29459.5,90.48,2
FAC00008,Denver,CO,Pickup,31,26,71.48,2216,43591.87,83.87,5
FAC00038,Detroit,MI,Pickup,30,25,79.8,2394,76899.12,83.33,5
FAC00018,Atlanta,GA,Pickup,62,51,66.02,4093,80854.03,82.26,11
FAC00010,Atlanta,GA,Pickup,53,43,64.83,3436,67412.83,81.13,10
FAC00028,Phoenix,AZ,Pickup,95,77,71.08,6753,326399.44,81.05,18
FAC00014,Denver,CO,Pickup,26,21,88.27,2295,35736.17,80.77,5
FAC00044,Minneapolis,MN,Pickup,92,74,80.76,7430,308677.06,80.43,18
FAC00028,Memphis,TN,Pickup,65,52,75.17,4886,149356.76,80.0,13
FAC00046,Denver,CO,Pickup,30,24,58.57,1757,41280.57,80.0,6


In [0]:
display(delivery_metrics)

facility_id,location_city,location_state,event_type,total_events,on_time_events,avg_detention_minutes,total_detention_minutes,total_revenue,on_time_rate_pct,delayed_events
FAC00020,Miami,FL,Pickup,113,72,68.88,7784,410731.56,63.72,41
FAC00002,Charlotte,NC,Delivery,105,44,95.53,10031,361623.07,41.9,61
FAC00002,Seattle,WA,Delivery,171,67,99.69,17047,717122.94,39.18,104
FAC00039,Portland,OR,Pickup,94,67,72.17,6784,253405.68,71.28,27
FAC00002,Las Vegas,NV,Pickup,114,70,79.79,9096,296540.21,61.4,44
FAC00044,Los Angeles,CA,Delivery,180,80,106.21,19117,630710.47,44.44,100
FAC00020,Kansas City,MO,Delivery,115,48,116.98,13453,261863.73,41.74,67
FAC00030,Portland,OR,Delivery,135,62,114.28,15428,721444.43,45.93,73
FAC00037,Atlanta,GA,Pickup,40,20,94.68,3787,52159.82,50.0,20
FAC00025,Miami,FL,Delivery,62,38,93.18,5777,155429.55,61.29,24


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

delivery_metrics.groupBy(
    "facility_id",
    "event_type"
).count().filter(
    F.col("count") > 1
).show()

+-----------+----------+-----+
|facility_id|event_type|count|
+-----------+----------+-----+
|   FAC00034|    Pickup|   18|
|   FAC00046|  Delivery|   19|
|   FAC00015|    Pickup|   18|
|   FAC00050|  Delivery|   19|
|   FAC00001|    Pickup|   18|
|   FAC00022|  Delivery|   19|
|   FAC00029|    Pickup|   18|
|   FAC00037|  Delivery|   19|
|   FAC00046|    Pickup|   18|
|   FAC00043|    Pickup|   18|
|   FAC00021|  Delivery|   19|
|   FAC00006|    Pickup|   18|
|   FAC00001|  Delivery|   19|
|   FAC00004|  Delivery|   19|
|   FAC00028|  Delivery|   19|
|   FAC00020|    Pickup|   18|
|   FAC00014|  Delivery|   19|
|   FAC00016|    Pickup|   18|
|   FAC00032|  Delivery|   19|
|   FAC00021|    Pickup|   18|
+-----------+----------+-----+
only showing top 20 rows


In [0]:
delivery_metrics.select(
    "facility_id",
    "location_city",
    "location_state",
    "event_type"
).show(20, False)

+-----------+-------------+--------------+----------+
|facility_id|location_city|location_state|event_type|
+-----------+-------------+--------------+----------+
|FAC00034   |Houston      |TX            |Pickup    |
|FAC00046   |Detroit      |MI            |Delivery  |
|FAC00015   |Kansas City  |MO            |Pickup    |
|FAC00050   |Indianapolis |IN            |Delivery  |
|FAC00001   |Columbus     |OH            |Pickup    |
|FAC00022   |Portland     |OR            |Delivery  |
|FAC00029   |Phoenix      |AZ            |Pickup    |
|FAC00037   |Denver       |CO            |Delivery  |
|FAC00046   |Houston      |TX            |Pickup    |
|FAC00050   |Seattle      |WA            |Delivery  |
|FAC00043   |Chicago      |IL            |Pickup    |
|FAC00021   |Portland     |OR            |Delivery  |
|FAC00006   |Las Vegas    |NV            |Pickup    |
|FAC00001   |Kansas City  |MO            |Delivery  |
|FAC00046   |Philadelphia |PA            |Pickup    |
|FAC00004   |Seattle      |W

In [0]:
spark.sql("""
DROP TABLE IF EXISTS workspace.logistics_gold.delivery_performance
""")

DataFrame[]

In [0]:
delivery_metrics.write \
.format("delta") \
.saveAsTable(
    "workspace.logistics_gold.delivery_performance"
)

In [0]:
#create gold table
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|customer_analysis   |false      |
|logistics_gold|delivery_performance|false      |
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|fuel_efficiency     |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
|logistics_gold|safety_analysis     |false      |
+--------------+--------------------+-----------+



In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.delivery_performance"
)

gold_table.alias("target").merge(
    delivery_metrics.alias("source"),
    """
    target.facility_id = source.facility_id
    AND target.location_city = source.location_city
    AND target.location_state = source.location_state
    AND target.event_type = source.event_type
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#Verify Gold Table Data
display(
    spark.table(
        "workspace.logistics_gold.delivery_performance"
    )
)

facility_id,location_city,location_state,event_type,total_events,on_time_events,avg_detention_minutes,total_detention_minutes,total_revenue,on_time_rate_pct,delayed_events
FAC00020,Miami,FL,Pickup,113,72,68.88,7784,410731.56,63.72,41
FAC00002,Charlotte,NC,Delivery,105,44,95.53,10031,361623.07,41.9,61
FAC00002,Seattle,WA,Delivery,171,67,99.69,17047,717122.94,39.18,104
FAC00039,Portland,OR,Pickup,94,67,72.17,6784,253405.68,71.28,27
FAC00002,Las Vegas,NV,Pickup,114,70,79.79,9096,296540.21,61.4,44
FAC00044,Los Angeles,CA,Delivery,180,80,106.21,19117,630710.47,44.44,100
FAC00020,Kansas City,MO,Delivery,115,48,116.98,13453,261863.73,41.74,67
FAC00030,Portland,OR,Delivery,135,62,114.28,15428,721444.43,45.93,73
FAC00037,Atlanta,GA,Pickup,40,20,94.68,3787,52159.82,50.0,20
FAC00025,Miami,FL,Delivery,62,38,93.18,5777,155429.55,61.29,24


In [0]:
#Check Total Records
spark.table(
    "workspace.logistics_gold.delivery_performance"
).count()

1850

In [0]:
#Check Table Exists
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|customer_analysis   |false      |
|logistics_gold|delivery_performance|false      |
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|fuel_efficiency     |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
|logistics_gold|safety_analysis     |false      |
+--------------+--------------------+-----------+



In [0]:
#View Top Records
spark.table(
    "workspace.logistics_gold.delivery_performance"
).show(10, False)

+-----------+-------------+--------------+----------+------------+--------------+---------------------+-----------------------+-------------+----------------+--------------+
|facility_id|location_city|location_state|event_type|total_events|on_time_events|avg_detention_minutes|total_detention_minutes|total_revenue|on_time_rate_pct|delayed_events|
+-----------+-------------+--------------+----------+------------+--------------+---------------------+-----------------------+-------------+----------------+--------------+
|FAC00020   |Miami        |FL            |Pickup    |113         |72            |68.88                |7784                   |410731.56    |63.72           |41            |
|FAC00002   |Charlotte    |NC            |Delivery  |105         |44            |95.53                |10031                  |361623.07    |41.9            |61            |
|FAC00002   |Seattle      |WA            |Delivery  |171         |67            |99.69                |17047                  |717